# 02 - FinalDecision classifier (Service vs Standby)
Thin notebook over `src.prediction.train_classifier`. Original: notebook cell 8, which crashed with `KeyError: 'cum_mileage'`
and therefore **has no recorded result**. Changes vs that cell (leakage fix, `Depot`, `Balanced` label, scaler fitting) are listed in `docs/methodology.md`.

Needs PyTorch. On the synthetic sample the metrics are meaningless.

In [ ]:
import logging
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebooks/ -> repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

import pandas as pd
from src import config as cfg
from src.data.loader import load_raw_dataset, resolve_data_path

# Real data if data/raw/kochi_induction_dataset.csv exists, else the SYNTHETIC sample.
USE_SAMPLE = not cfg.RAW_DATA_PATH.exists()
path, SOURCE = resolve_data_path(use_sample=USE_SAMPLE)
df = load_raw_dataset(path)
print("data source:", SOURCE, "| shape:", df.shape)

In [ ]:
import dataclasses
from src.prediction.train_classifier import train_classifier

config = dataclasses.replace(cfg.ClassifierTrainConfig(), epochs=20)    # notebook value: 20
classifier, report = train_classifier(df, config, report_dir=cfg.DEFAULT_RUN_DIR, data_source=SOURCE)

In [ ]:
print("features used:", report["features"])
print("split sizes:", report["rows"])
pd.DataFrame(report["metrics"]).loc[["accuracy", "precision_service", "recall_service", "f1_service", "roc_auc", "majority_baseline_accuracy"]]

Compare accuracy with `majority_baseline_accuracy`: a classifier that always predicts the majority class already scores that.

In [ ]:
from src.evaluation.visualization import plot_training_history, plot_confusion_matrix
plot_training_history(report["history"], "Classifier training", "BCE loss")
plot_confusion_matrix(report["metrics"]["test"]["confusion_matrix"], title="Test confusion matrix");

## Optional: chronological split
Compares against the random split above.

In [ ]:
chrono = dataclasses.replace(config, split_strategy="chronological")
_, report_chrono = train_classifier(df, chrono, save=False, data_source=SOURCE)
report_chrono["metrics"]["test"]["accuracy"], report["metrics"]["test"]["accuracy"]